# Stage 4 walkthrough: retrieval, built by deliberate failure

One section per checkpoint, added as each one finishes. Each section is safe to rerun on
its own after the **Setup** cell. Like the Stage 2→3 notebook, every cell calls the
project's own functions and reimplements nothing, so what you see is what `fc` does.

| Checkpoint | Section | What it shows |
|---|---|---|
| **0** | Eval set, before any search | What the corpus can answer; whether `questions.yaml` is well-formed |
| A | Vector vs BM25 | The probe queries failing, vector-only first |
| B | RRF | What fusion fixed and what it didn't |
| C | Eval harness | recall@k / MRR / nDCG@10, unjudged@10, adjudication |
| D | Chunker A/B | The 3 × 2 grid, label agreement, the verdict |
| E | Strategies | Global top-k concentration vs `fan_out_peers` coverage |

**Prerequisites**
- The `.venv` kernel (`uv sync --group notebook`)
- `fc embed` and `fc build-index` already run for both chunkers (the manifests are read below)
- From Checkpoint A on: OpenSearch up (`docker ps`) and Ollama running `nomic-embed-text`

Plan: `~/.claude/plans/provide-me-a-phased-fluttering-puzzle.md` · Rubric and questions:
`evals/retrieval/`.

## Setup

`Settings` is never printed: it holds your EDGAR User-Agent contact, and this repo is public.

In [1]:
import os
from collections import Counter
from pathlib import Path

# .env, config/, data/ and evals/ are relative to the repo root, exactly as when running `fc`.
if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd
import yaml

from filing_copilot.config import get_settings
from filing_copilot.embed import NOMIC_EMBED_TEXT
from filing_copilot.filings.chunkers import ITEM_TITLES
from filing_copilot.index import manifest_path, read_manifest
from filing_copilot.structured import Corpus

settings = get_settings()
corpus = Corpus.load(settings.corpus_path)

EVALS = Path("evals/retrieval")
CHUNKERS = ("item_aware", "fixed_window")

manifests = {
    chunker: read_manifest(manifest_path(settings.manifests_dir, chunker, NOMIC_EMBED_TEXT.cache_key))
    for chunker in CHUNKERS
}
for chunker, rows in manifests.items():
    print(f"{chunker:>12}: {len(rows):,} chunks")

  item_aware: 16,370 chunks
fixed_window: 19,763 chunks


---
# Checkpoint 0 · The eval set, written before any search code

**Why this comes first.** Every retrieval metric measures agreement with *your* relevance
judgments. Once you have watched the retriever fail (Checkpoint A), you know what it is bad
at, and questions written afterwards lean toward that, deliberately or not. So the questions
are written now, while you still know nothing about how the retriever behaves.

Nothing here searches. It answers two questions:
1. **What can the corpus answer?** Questions about a 10-Q, or about FY2022, have no gold to
   find.
2. **Is `questions.yaml` well-formed?** The package's own validator (`retrieval/evalset.py`),
   the same one the eval runs.

## 0.1 · What the corpus holds

Chunks per filer and fiscal year. Every cell here is one 10-K. A small count flags a filing
whose text is mostly elsewhere (an EX-13 exhibit, a referral) or mostly tables.

In [13]:
rows = manifests["item_aware"]
by_filing = pd.DataFrame(
    [(r.ticker, r.fiscal_year) for r in rows], columns=["ticker", "fiscal_year"]
).value_counts().unstack(fill_value=0)
print("forms:", dict(Counter(r.form for r in rows)))
by_filing

forms: {'10-K': 16370}


fiscal_year,2025,2024,2023
ticker,,,
COF,345,313,300
SYF,232,240,231
AXP,251,252,239
BFH,233,232,211
ALLY,362,383,375
SOFI,445,432,405
JPM,433,438,456
BAC,333,328,337
C,369,411,410


## 0.2 · Which items each filer's text was sectioned into

`items` is what the `scope.items` field of a question refers to. It is also what the
Phase 6 `section_targeted` filter matches. An item missing for a filer means a question
scoped to it has nowhere to look. That is a Stage 2 gap, not a retrieval failure, so it is
worth knowing before you write a question that depends on it.

Chunk counts per (ticker, item), summed over the three years. Chunks claimed by several
items count once under each.

In [14]:
item_counts = Counter((r.ticker, item) for r in rows for item in r.items)
by_item = (
    pd.Series(item_counts)
    .unstack(fill_value=0)
    .reindex(columns=[i for i in ITEM_TITLES if any(i == k[1] for k in item_counts)], fill_value=0)
)
by_item

,1,1A,1C,2,3,5,7,7A,8,9A
AAPL,21,98,2,3,6,3,20,3,57,6
ALLY,111,175,0,3,0,3,410,3,375,3
AXP,123,152,13,3,0,6,200,0,217,3
BAC,52,175,0,3,0,3,333,3,393,3
BFH,124,212,9,3,3,5,107,0,0,24
C,187,141,49,0,52,26,342,514,621,12
CFG,97,85,8,3,0,3,141,3,233,3
COF,110,195,15,3,3,3,258,325,0,4
FITB,68,124,12,3,3,6,263,3,386,10
JEF,55,71,6,3,3,3,164,3,329,3


## 0.3 · Check `questions.yaml`

`validate_questions` is the same check `fc eval-retrieval` runs. It lists problems rather
than raising, so one typo does not hide the rest. The mix table at the end compares against
the file's target mix (lexical 7 · semantic 7 · entity 10 · peer 6).

In [15]:
from filing_copilot.retrieval.evalset import CorpusVocabulary, validate_questions

# The same validation `fc eval-retrieval` runs -- moved into the package at Checkpoint C.
vocabulary = CorpusVocabulary(
    tickers=frozenset(c.ticker for c in corpus),
    fiscal_years=frozenset(r.fiscal_year for r in rows),
    items=frozenset(ITEM_TITLES),
)
raw = yaml.safe_load((EVALS / "questions.yaml").read_text())
problems = validate_questions(raw, vocabulary)
questions = [q for q in raw["questions"] or [] if isinstance(q, dict)]

print(f"{len(questions)} questions, {len(problems)} problems")
for p in problems:
    print("  -", p)

if questions:
    df = pd.DataFrame(questions)
    print(f"\nblind share: {(df['split'] == 'blind').mean():.0%}  (target ~20%)")
    display(pd.crosstab(df["archetype"], df["persona"], margins=True))

30 questions, 0 problems

blind share: 20%  (target ~20%)


persona,aml,card_intelligence,credit_risk,model_risk,All
archetype,,,,,
entity,2,3,4,1,10
lexical,0,1,2,4,7
peer,2,2,1,1,6
semantic,1,2,3,1,7
All,5,8,10,7,30


## 0.4 · Is the rubric frozen?

Labeling (Checkpoint C) should not start until every edge case in `RUBRIC.md` has a
decision and the file carries a *Frozen on* date.

In [16]:
rubric = (EVALS / "RUBRIC.md").read_text()
undecided = rubric.count("- Decision:\n")
frozen_line = next(line for line in rubric.splitlines() if line.startswith("**Frozen on:**"))
print(f"edge cases without a decision: {undecided}")
print(frozen_line)

edge cases without a decision: 0
**Frozen on:** 2026-09-24


---
# Checkpoint A · Vector vs BM25, each on its own

**What this checkpoint is for.** It is the plan's single most instructive failure
(`PROJECT_PLAN.txt` §1.9). Run each retriever *alone* on the same probe queries, vector
first, and write down how each one fails. Phase 3 fuses them, and the reasons fusion
helps should be things you saw here.

The notebook calls exactly what `fc search` calls:

```
fc search "material weakness" --mode vector
fc search "material weakness" --mode bm25
fc search "material weakness" --compare
```

**Needs:** OpenSearch up (`docker ps`) and Ollama running `nomic-embed-text`.

In [17]:
from dataclasses import replace

from IPython.display import display

from filing_copilot.embed import OllamaEncoder
from filing_copilot.index import OpenSearchClient, index_name
from filing_copilot.retrieval import (
    Hit,
    bm25_search,
    cosine_from_knn_score,
    encode_query,
    vector_search,
)

search_client = OpenSearchClient(settings.opensearch_url)
encoder = OllamaEncoder(host=settings.ollama_host, model=NOMIC_EMBED_TEXT)
INDEX = {chunker: index_name(settings.opensearch_index_prefix, chunker) for chunker in CHUNKERS}
print("OpenSearch", search_client.ping(), "|", INDEX)


def run(query: str, mode: str, *, chunker: str = "item_aware", k: int = 5) -> list[Hit]:
    """One retriever, one query -- the same calls `fc search` makes."""
    if mode == "vector":
        return vector_search(search_client, INDEX[chunker], encode_query(encoder, query), k=k)
    return bm25_search(search_client, INDEX[chunker], query, k=k)


def table(hits: list[Hit], mode: str, snippet: int = 120) -> pd.DataFrame:
    """Hits as a table. Vector scores are shown as cosine; BM25 scores are raw."""
    return pd.DataFrame(
        {
            "rank": h.rank,
            ("cosine" if mode == "vector" else "bm25"): round(
                cosine_from_knn_score(h.score) if mode == "vector" else h.score, 3
            ),
            "ticker": h.ticker,
            "FY": h.fiscal_year,
            "items": ",".join(h.items),
            "text": " ".join(h.text.split())[:snippet],
        }
        for h in hits
    ).set_index("rank")


pd.set_option("display.max_colwidth", 130)

OpenSearch 2.19.1 | {'item_aware': 'filings-item_aware', 'fixed_window': 'filings-fixed_window'}


## A.1 · The probe queries

Short, deliberately varied queries. They are probes, not eval questions: they show *how*
each retriever fails, and the eval (Checkpoint C) measures *how often*. Edit freely.

**Kept clear of the blind split** (r006, r009, r014, r015, r017, r025). Running a blind
question here would spoil it. So there is nothing on JPM/AML, AML consent orders, the
late-fee rule, Item 1C, USB bond losses, or SYF allowance models.

In [18]:
PROBES = {
    "material weakness": "exact regulatory phrase",
    "SYF": "a bare ticker",
    "CECL": "an acronym",
    "Item 9A": "an item number",
    "customer concentration": "a term with a common-word twin",
    "Durbin Amendment": "a named regulation",
    "what happens if a big retail partner stops working with the company": "paraphrase",
    "banks worried that depositors will pull their money": "paraphrase",
}
for q, why in PROBES.items():
    print(f"{why:>32}:  {q}")

         exact regulatory phrase:  material weakness
                   a bare ticker:  SYF
                      an acronym:  CECL
                  an item number:  Item 9A
  a term with a common-word twin:  customer concentration
              a named regulation:  Durbin Amendment
                      paraphrase:  what happens if a big retail partner stops working with the company
                      paraphrase:  banks worried that depositors will pull their money


## A.2 · Step 1: vector only, and watch it fail

Top 5 per probe, from the `item_aware` index. For each probe, ask:
- Is the top hit about the *subject* of the query, or just near it in meaning?
- For "SYF", are the hits Synchrony's? (Hold that thought for A.5.)
- How far apart are the cosines of rank 1 and rank 5? (Hold that for A.4.)

In [19]:
vector_results = {q: run(q, "vector") for q in PROBES}
for q, hits in vector_results.items():
    print(f"\n=== {q!r}  ({PROBES[q]})")
    display(table(hits, "vector"))


=== 'material weakness'  (exact regulatory phrase)


,cosine,ticker,FY,items,text
rank,,,,,
1,0.614,RF,2025,1A,ion. Weakness in commodity businesses could adversely affect our performance. Many of our borrowers operate in industrie
2,0.598,BFH,2024,1A,"onomic slowdown, economic and market volatility, and other adverse economic conditions, including persistent inflation,"
3,0.595,WFC,2024,"1A,7","dition, weak or deteriorating economic conditions make it more challenging for us to increase our consumer and commercia"
4,0.595,WFC,2023,"1A,7","dition, weak or deteriorating economic conditions make it more challenging for us to increase our consumer and commercia"
5,0.593,BFH,2025,1A,"onomic slowdown, economic and market volatility, and other adverse economic conditions, including persistent inflation,"



=== 'SYF'  (a bare ticker)


,cosine,ticker,FY,items,text
rank,,,,,
1,0.682,SYF,2025,1,"fer and deliver our credit products instantly to customers across multiple channels. We continue to invest in, and devel"
2,0.680,SYF,2024,1,"s to offer and deliver our credit products instantly to customers across multiple channels. We continue to invest in, an"
3,0.678,SYF,2025,1,"our customers’ outstanding balances, based on a number of factors, including pricing (interest rates and fees), product"
4,0.674,SYF,2025,1,"primarily national and regional health, wellness and veterinary providers, such as Aspen Dental, Mars Petcare, and other"
5,0.668,SYF,2023,1,ch allow us to offer and deliver our credit products instantly to customers across multiple channels. We continue to inv



=== 'CECL'  (an acronym)


...

,cosine,ticker,FY,items,text
rank,,,,,
1,0.646,SOFI,2025,1,"rs using practices that are deemed to be unfair, deceptive or abusive. Durbin Amendment. Section 1075 of the Dodd-Frank"
2,0.625,SOFI,2023,1,"ervices to consumers using practices that are deemed to be unfair, deceptive or abusive. Section 1075 of the Dodd-Frank"
3,0.593,MA,2025,1,eral Reserve in Regulation II (related to the Durbin Amendment to the Dodd-Frank Wall Street Reform and Consumer Protect
4,0.591,V,2025,1A,trict Court for the District of North Dakota ruled that the Federal Reserve exceeded its authority in implementing Regul
5,0.585,SOFI,2025,1A,"le, Galileo’s provision of technology services to SoFi Bank is subject to Regulation W, including the requirement that t"



=== 'what happens if a big retail partner stops working with the company'  (paraphrase)


,cosine,ticker,FY,items,text
rank,,,,,
1,0.709,SYF,2024,1A,few programs with national and regional retailer and manufacturer partners also may be terminated at will by the partner
2,0.706,SYF,2025,1A,few programs with national and regional retailer and manufacturer partners also may be terminated at will by the partner
3,0.684,SYF,2023,1A,revenue is generated from the credit products we provide to customers of our partners pursuant to program agreements we
4,0.677,SYF,2024,1A,"ition, if our partners engage in improper business practices, do not adhere to the terms of our program agreements or ot"
5,0.675,AXP,2025,1A,"r cobrand partners, which can be significant, based primarily on the amount of Card Member spending and corresponding re"



=== 'banks worried that depositors will pull their money'  (paraphrase)


,cosine,ticker,FY,items,text
rank,,,,,
1,0.746,USB,2024,1A,"ed institutions. Further, even if governmental support for financial institutions is available in the future, it may not"
2,0.744,USB,2025,1A,ny raises the interest rates it pays on deposits to avoid losing deposits to competitors or because the Company loses de
3,0.741,USB,2023,1A,collateralized deposits could cause rapid and significant deposit outflows. This risk is exacerbated by technological de
4,0.741,USB,2023,1A,pany to increase the interest rates it pays on deposits. If the Company’s competitors raise the interest rates they pay
5,0.723,BFH,2025,1A,"pede our competitive position in the deposits market. Furthermore, the failures of other financial institutions (such as"


### Your observations, vector only

Write these **before** running A.3. Once you have seen BM25's results, the vector-only
failures will be harder to describe on their own terms.

| Probe | What came back | Failed? How? |
|---|---|---|
| material weakness | | |
| SYF | | |
| CECL | | |
| Item 9A | | |
| customer concentration | | |
| Durbin Amendment | | |
| retail partner paraphrase | | |
| depositor paraphrase | | |

## A.3 · Step 2: add BM25, same probes

Same queries, same index, lexical scoring. The summary table puts the two side by side:
how many of the top 5 they share, and which companies each one returned. The full BM25
hits follow.

In [20]:
bm25_results = {q: run(q, "bm25") for q in PROBES}

summary = pd.DataFrame(
    {
        "why": PROBES[q],
        "shared top-5": len(
            {h.chunk_id for h in vector_results[q]} & {h.chunk_id for h in bm25_results[q]}
        ),
        "vector companies": ", ".join(sorted({h.ticker for h in vector_results[q]})),
        "bm25 companies": ", ".join(sorted({h.ticker for h in bm25_results[q]})) or "(no hits)",
    }
    for q in PROBES
).set_axis(list(PROBES))
display(summary)

for q, hits in bm25_results.items():
    print(f"\n=== {q!r}  ({PROBES[q]})")
    display(table(hits, "bm25") if hits else "no hits")

,why,shared top-5,vector companies,bm25 companies
material weakness,exact regulatory phrase,0,"BFH, RF, WFC","SOFI, WFC"
SYF,a bare ticker,0,SYF,SYF
CECL,an acronym,2,"BAC, COF",COF
Item 9A,an item number,0,"AAPL, BAC","USB, WFC"
customer concentration,a term with a common-word twin,0,"BFH, MA, SYF","AXP, BAC"
Durbin Amendment,a named regulation,2,"MA, SOFI, V","MA, SOFI, V"
what happens if a big retail partner stops working with the company,paraphrase,0,"AXP, SYF","SOFI, SYF"
banks worried that depositors will pull their money,paraphrase,0,"BFH, USB","RF, SOFI, TFC, WFC"



=== 'material weakness'  (exact regulatory phrase)


,bm25,ticker,FY,items,text
rank,,,,,
1,9.481,SOFI,2025,1A,al claims or other claims arising from the restatement and material weaknesses in our internal control over financial re
2,9.339,SOFI,2024,1A,and maintain effective internal controls over financial reporting could adversely affect the results of assessments by o
3,9.295,SOFI,2023,1A,and maintain effective internal controls over financial reporting could adversely affect the results of assessments by o
4,9.113,WFC,2025,"1A,7",and because it is likely that materially different amounts would be reported under different conditions or using differe
5,9.113,WFC,2023,"1A,7",and because it is likely that materially different amounts would be reported under different conditions or using differe



=== 'SYF'  (a bare ticker)


,bm25,ticker,FY,items,text
rank,,,,,
1,8.451,SYF,2025,5,"Table of Contents Market for Registrant's Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Secu"
2,8.451,SYF,2024,5,"Market for Registrant's Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities ___________"
3,8.451,SYF,2023,5,"Market for Registrant's Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities ___________"


...

,bm25,ticker,FY,items,text
rank,,,,,
1,13.876,SOFI,2025,1,"rs using practices that are deemed to be unfair, deceptive or abusive. Durbin Amendment. Section 1075 of the Dodd-Frank"
2,13.160,SOFI,2025,1,"eatures and may not exceed certain limitations on the amount of credit extended to such persons, individually and in the"
3,13.160,MA,2023,8,"ng the plaintiffs’ request for class certification and in January 2021, the Network Defendants’ request for permission t"
4,12.275,SOFI,2025,1A,"le, Galileo’s provision of technology services to SoFi Bank is subject to Regulation W, including the requirement that t"
5,11.171,V,2023,8,"eral district court in Texas, alleging that Visa has, among other things, monopolized and attempted to monopolize debit"



=== 'what happens if a big retail partner stops working with the company'  (paraphrase)


,bm25,ticker,FY,items,text
rank,,,,,
1,17.950,SYF,2023,1,"3, the length of our relationship with each of our five largest partners was over 10 years, and in the case of Lowe's, 4"
2,17.526,SOFI,2025,1,Business Company Overview We are a mission driven company designed to help our members achieve financial independence in
3,17.101,SYF,2024,1,"as Generac, Nationwide Marketing Group and the Home Furnishings Association. At December 31, 2024, the length of our rel"
4,16.942,SOFI,2024,1,Business Company Overview We are a mission driven company designed to help our members achieve financial independence in
5,15.908,SYF,2023,8,d useful lives. Defined-lived intangible assets are evaluated for impairment whenever events or changes in circumstances



=== 'banks worried that depositors will pull their money'  (paraphrase)


,bm25,ticker,FY,items,text
rank,,,,,
1,11.052,WFC,2025,1,"n, and future prospects including current and projected capital ratios and levels, the competence, experience, and integ"
2,11.052,WFC,2024,1,"n, and future prospects including current and projected capital ratios and levels, the competence, experience, and integ"
3,10.123,TFC,2025,1,"ory actions, cease and desist orders, civil money penalties, or other actions. In October 2025, the FDIC and the OCC iss"
4,10.122,SOFI,2025,1,"ilable to the federal banking regulators include, among other things, the ability to issue cease and desist or removal o"
5,9.933,RF,2023,1,"nk or savings and loan association, if after such acquisition, the BHC will directly or indirectly own or control 5% or"


### Your observations, BM25

| Probe | What changed vs vector | Which retriever was right? |
|---|---|---|
| material weakness | | |
| SYF | | |
| CECL | | |
| Item 9A | | |
| customer concentration | | |
| Durbin Amendment | | |
| retail partner paraphrase | | |
| depositor paraphrase | | |

## A.4 · Mechanism: what a vector score is, and why a cosine alone says little

The index scores by inner product. Every vector is unit length, so inner product *is*
cosine. OpenSearch does not return it raw, though: faiss reports `1 + ip` when `ip ≥ 0`
and `1 / (1 − ip)` below that, so every score is positive. `cosine_from_knn_score`
undoes this.

Below: a chunk searched with its own vector scores exactly **2.0** (ip = 1). Then the
cosine between random pairs of chunks. On this corpus, **unrelated chunks are not near
0**. Every 10-K chunk shares register, vocabulary and boilerplate, so the whole corpus
sits in a narrow cone.

That is why a vector retriever always returns *something* that looks confident, and why a
fixed cosine threshold (for example "relevant if > 0.7") would mean nothing here. Scores
are only comparable within one query's result list. The same property is what makes RRF
(Phase 3) rank-based.

In [10]:
import random

sample = search_client.search(
    INDEX["item_aware"],
    {"size": 400, "_source": ["chunk_id", "embedding"],
     "query": {"function_score": {"random_score": {"seed": 7, "field": "_seq_no"}}}},
)["hits"]["hits"]
vectors = [h["_source"]["embedding"] for h in sample]

own = vector_search(search_client, INDEX["item_aware"], vectors[0], k=1)[0]
print(f"a chunk's own vector: score {own.score:.4f} -> cosine {cosine_from_knn_score(own.score):.4f}")

rng = random.Random(0)
pairs = pd.Series(
    [sum(a * b for a, b in zip(*rng.sample(vectors, 2), strict=True)) for _ in range(3000)]
)
print("\ncosine between 3,000 random chunk pairs:")
display(pairs.describe(percentiles=[0.05, 0.5, 0.95]).round(3).to_frame("cosine").T)

a chunk's own vector: score 2.0000 -> cosine 1.0000

cosine between 3,000 random chunk pairs:


,count,mean,std,min,5%,50%,95%,max
cosine,3000.0,0.72,0.053,0.548,0.641,0.717,0.814,0.968


## A.5 · Mechanism: the two retrievers do not see the same text

Each vector was embedded from `contextual_prefix + chunk`. The prefix names the company,
**ticker**, form, period and item, for example:

```
Synchrony Financial (SYF) · 10-K · period ending 2025-12-31 · Item 1A. Risk Factors
```

BM25 indexes only the chunk `text`, never the prefix. So for a ticker query, the vector
side has the ticker in **every** chunk of that filer, and BM25 has it only where the prose
happens to use it. This is the reverse of the plan's textbook failure ("vector search
misses exact tickers"), and it comes from a Stage 2 design choice.

It also shows why "SYF" is really a **filter**, not a query (Phase 6).

In [11]:
TICKER = "SYF"
filer_rows = [r for r in manifests["item_aware"] if r.ticker == TICKER]
in_text = sum(TICKER in r.text for r in filer_rows)
print(f"{TICKER} chunks: {len(filer_rows):,}")
print(f"  with '{TICKER}' in the embedded input (prefix): {len(filer_rows):,}  (all of them)")
print(f"  with '{TICKER}' in the BM25-indexed text:      {in_text:,}")

K = 20
for mode in ("vector", "bm25"):
    hits = run(TICKER, mode, k=K)
    theirs = sum(h.ticker == TICKER for h in hits)
    # Share of k, not of what came back: BM25 can only return chunks containing the term.
    print(f"{mode:>6}: {len(hits):>2} of {K} requested came back; {theirs} are {TICKER}'s")

SYF chunks: 703
  with 'SYF' in the embedded input (prefix): 703  (all of them)
  with 'SYF' in the BM25-indexed text:      3
vector: 20 of 20 requested came back; 20 are SYF's
  bm25:  3 of 20 requested came back; 3 are SYF's


## A.6 · Mechanism: the task prefix, measured rather than assumed

`CLAUDE.md` names this trap: nomic needs `search_query:` on questions, and omitting it
degrades retrieval *silently*. `encode_query` always adds it. Here we deliberately skip it,
using a model identical except `needs_task_prefix=False`, and compare:
- how far the two query vectors are from each other, and
- how much the top 10 changes.

In [12]:
unprefixed = OllamaEncoder(
    host=settings.ollama_host, model=replace(NOMIC_EMBED_TEXT, needs_task_prefix=False)
)
rows_ = []
for q in ["material weakness", "customer concentration", "banks worried that depositors will pull their money"]:
    with_prefix = encode_query(encoder, q)
    without = encode_query(unprefixed, q)
    a = {h.chunk_id for h in vector_search(search_client, INDEX["item_aware"], with_prefix, k=10)}
    b = {h.chunk_id for h in vector_search(search_client, INDEX["item_aware"], without, k=10)}
    rows_.append({
        "query": q,
        "cos(prefixed, bare)": round(sum(x * y for x, y in zip(with_prefix, without, strict=True)), 3),
        "top-10 overlap": f"{len(a & b)}/10",
    })
unprefixed.close()
display(pd.DataFrame(rows_).set_index("query"))

,"cos(prefixed, bare)",top-10 overlap
query,,
material weakness,0.955,7/10
customer concentration,0.927,5/10
banks worried that depositors will pull their money,0.970,9/10


## A.7 · Bring to the Checkpoint B discussion

1. **Where vector-only failed, and how.** Topic drift? Matching a common-word twin
   ("weakness" as in economic weakness)?
2. **Where BM25 failed.** Paraphrases with no shared words? A word matched without its
   meaning?
3. **Did any probe fail the way the plan predicted, or the reverse?** (Think about A.5.)
4. **Any probe where both were wrong.** Fusion cannot fix that. Note it for the eval.

Optional: rerun A.2 and A.3 with `chunker="fixed_window"`. Don't draw conclusions yet:
the chunker A/B is Phase 5, on the eval set, not on eight probes.

---
# Checkpoint B · Reciprocal Rank Fusion

Each retriever failed differently in Checkpoint A. RRF combines them using **ranks
alone**: each chunk scores

$$\text{RRF}(d) = \sum_{\text{lists}} \frac{1}{k + \text{rank}(d)}, \qquad k = 60$$

A list that did not return the chunk contributes nothing. There are no score scales to
reconcile. A.4 showed why that matters: a cosine of 0.72 is what *unrelated* chunks score
here, and BM25 scores are unbounded, so any score blend would need a normalization, which
is one more knob to tune.

**k = 60 is fixed** (ADR-0010) until the eval harness (Checkpoint C) can measure a
change. So is the candidate depth: each retriever returns its top **100** before fusion.

```
fc search "material weakness" --mode hybrid
fc search "material weakness" --compare      # all three
```

**Needs:** the Setup cell and the first Checkpoint A cell (clients, `run`, `table`, `PROBES`).

## B.1 · What k does, worked by hand (no server)

The contribution of one list at rank *r* is `1/(60 + r)`. The comparison that matters is
a chunk at rank 1 in **one** list versus a chunk at rank *r* in **both**. Agreement wins
whenever `2/(60 + r) > 1/61`, which means for any *r* up to 61.

So a chunk both retrievers placed around 50th beats a chunk only one of them placed 1st.
That is the design: agreement is the signal. Hold that thought for B.3.

In [ ]:
from filing_copilot.retrieval import CANDIDATES, RRF_K, hybrid_search, reciprocal_rank_fusion

ranks = [1, 2, 5, 10, 30, 61, 62, 100]
display(pd.DataFrame({
    "rank r": ranks,
    "one list: 1/(k+r)": [round(1 / (RRF_K + r), 4) for r in ranks],
    "both lists: 2/(k+r)": [round(2 / (RRF_K + r), 4) for r in ranks],
    "beats rank 1 in one list?": [2 / (RRF_K + r) > 1 / (RRF_K + 1) for r in ranks],
}).set_index("rank r"))

In [ ]:
# The same thing through the real function, on toy lists: 'solo' is vector's #1 and
# absent from BM25; 'both' is 50th in each list.
def toy(prefix: str, n: int) -> list[str]:
    return [f"{prefix}{i}" for i in range(n)]

def as_hits(ids: list[str]) -> list[Hit]:
    template = run("CECL", "bm25", k=1)[0]  # any real hit, re-labelled
    return [replace(template, chunk_id=c, rank=r) for r, c in enumerate(ids, start=1)]

fused = reciprocal_rank_fusion({
    "vector": as_hits(["solo", *toy("v", 48), "both"]),
    "bm25": as_hits([*toy("b", 49), "both"]),
}, limit=3)
display(pd.DataFrame(
    {"rank": h.rank, "chunk": h.chunk_id, "rrf": round(h.score, 5), "sources": h.sources} for h in fused
).set_index("rank"))

## B.2 · The probes, three ways

For each probe: vector alone, BM25 alone, and the fused top 5. In the hybrid table,
`sources` shows where each list ranked the chunk, for example `v#6 b#1`. `v#-` means
vector's top 100 did not contain it.

In [ ]:
def provenance(h: Hit) -> str:
    ranks = dict(h.sources)
    return " ".join(f"{n[0]}#{ranks.get(n, '-')}" for n in ("vector", "bm25"))


def fused_table(hits: list[Hit], snippet: int = 120) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "rank": h.rank,
            "rrf": round(h.score, 4),
            "sources": provenance(h),
            "ticker": h.ticker,
            "FY": h.fiscal_year,
            "items": ",".join(h.items),
            "text": " ".join(h.text.split())[:snippet],
        }
        for h in hits
    ).set_index("rank")


def hybrid(query: str, *, chunker: str = "item_aware", k: int = 5) -> list[Hit]:
    return hybrid_search(search_client, INDEX[chunker], query, encode_query(encoder, query), k=k)


hybrid_results = {q: hybrid(q) for q in PROBES}
for q in PROBES:
    print(f"\n{'=' * 100}\n{q!r}  ({PROBES[q]})")
    print("-- vector"); display(table(vector_results[q], "vector"))
    print("-- bm25"); display(table(bm25_results[q], "bm25") if bm25_results[q] else "no hits")
    print("-- hybrid"); display(fused_table(hybrid_results[q]))

## B.3 · Where did the fused top 5 come from?

For each probe, how many fused hits came from **both** lists, from vector only, or from
BM25 only. Also, how many of each retriever's *own* top 5 survived into the fused top 5.

Three patterns to look for:
- **Fusion fixed it.** One retriever was wrong, the other right, and the right one's hits
  made it into the fused list.
- **Agreement over conviction.** One list's strong top hits were pushed out by chunks
  that *both* lists ranked only moderately. That is B.1's arithmetic, working against you.
  Check 'material weakness': BM25's genuine restatement chunks at #2 and #3 against
  consensus chunks ranked in the 20s by both.
- **Both wrong.** Fusion cannot fix a probe neither retriever gets right.

In [ ]:
def composition(q: str) -> dict[str, object]:
    fused = hybrid_results[q]
    both = sum(len(h.sources) == 2 for h in fused)
    only = {n: sum(dict(h.sources).keys() == {n} for h in fused) for n in ("vector", "bm25")}
    fused_ids = {h.chunk_id for h in fused}
    return {
        "why": PROBES[q],
        "from both": both,
        "vector only": only["vector"],
        "bm25 only": only["bm25"],
        "vector top-5 kept": len({h.chunk_id for h in vector_results[q]} & fused_ids),
        "bm25 top-5 kept": len({h.chunk_id for h in bm25_results[q]} & fused_ids),
        "deepest source rank": max(r for h in fused for _, r in h.sources),
    }

display(pd.DataFrame(composition(q) for q in PROBES).set_axis(list(PROBES)))

## B.4 · Why the candidate depth is 100, not k

Fusion can only promote what a list returned. If each retriever returned only its top 5,
a chunk at vector #21 / BM25 #28 (a consensus hit) would never be seen. The "deepest
source rank" column above shows how far down the fused top 5 actually reached.

Below, the same fused top 5 computed at depth 10 against depth 100. Where they differ,
consensus from deeper in the lists changed the answer. Candidate depth is a real
parameter, not a performance detail. Like k, it stays fixed until the eval can measure
it.

In [ ]:
rows_ = []
for q in PROBES:
    vec = encode_query(encoder, q)
    shallow = hybrid_search(search_client, INDEX["item_aware"], q, vec, k=5, candidates=10)
    deep = hybrid_results[q]
    rows_.append({
        "probe": q,
        "top-5 overlap, depth 10 vs 100": f"{len({h.chunk_id for h in shallow} & {h.chunk_id for h in deep})}/5",
    })
display(pd.DataFrame(rows_).set_index("probe"))

## B.5 · What this sets up

Eight probes show *how* fusion behaves. They cannot say whether hybrid is *better* than
vector-only. That needs labelled questions and metrics, which is Checkpoint C. It is also
where "agreement over conviction" gets measured, not just noticed.

Things **not** to do yet, per ADR-0010 and the plan:
- change k from 60, or weight one list above the other;
- change the candidate depth from 100;
- add a reranker.

Each is a legitimate lever. Each is also the "tuning by eyeballing" the plan warns against,
until there is a number to move.

Optional notes, if you want them for the interview story:

| Probe | Fusion fixed it / diluted it / both wrong | Evidence (sources column) |
|---|---|---|
| | | |

---
# Checkpoint C · The eval harness, and the human in the loop

Checkpoints A and B showed *how* each retriever behaves on eight probes. This checkpoint
measures *how often* it gets things right, on 30 questions, against labels you make by
reading the filings.

```
(Phase 0: rubric + questions) → label gold spans → run → adjudicate (blind) → tag failures
                                                   ↑                                │
                                                   └── change ONE thing, rerun dev ─┘
```

| Step | Who | Command |
|---|---|---|
| Label gold spans, from the filing | **you** | `fc label-span -t SYF --fy 2024 -i 1A -f "partners"` |
| Score every question × retriever × chunker | machine | `fc eval-retrieval` |
| Judge what the retrievers found that the labels missed | **you**, blind | `fc adjudicate` |
| Say why each miss happened | **you** | edit `evals/retrieval/failures.yaml` |

**Everything below runs whether or not any labels exist yet.** Until they do, the relevance
metrics are blank. Peer coverage, which needs no labels, is already real.

## C.1 · The eval set, as the harness loads it

`load_questions` validates everything, including the drift check below. `check_gold`
compares each gold span's `filing_sha256` with the manifest's `text_sha256`. If Stage 2's
normalization ever changes, offsets can silently point at different words, and this is
where that gets caught.

In [ ]:
from filing_copilot.retrieval.evalset import (
    check_gold, graded_spans, load_failures, load_judgments, load_questions, load_rechecks,
)

eval_questions = load_questions(EVALS / "questions.yaml", vocabulary)
judgments = load_judgments(EVALS / "judgments.yaml")
drift = check_gold(eval_questions, manifests["item_aware"])

print(f"{len(eval_questions)} questions · {sum(q.is_labelled for q in eval_questions)} labelled · "
      f"{len(judgments)} judgments · {len(drift)} drifted gold spans")
display(pd.DataFrame(
    {"id": q.id, "split": q.split, "archetype": q.archetype, "gold spans": len(q.gold),
     "adjudicated": sum(j.question == q.id for j in judgments), "question": q.question}
    for q in eval_questions
).set_index("id"))

## C.2 · Labeling: find the passage by reading the filing

`find_passages` is a plain substring search over the filing's normalized text. It is
deliberately **not** retrieval: labels drawn from what the retriever returned would
measure agreement with the retriever. Each match expands to its paragraph, and
`gold_entry` gives the exact YAML to paste under a question's `gold:`.

This is what `fc label-span` prints. Below: r001 (Synchrony partner concentration, dev),
FY2024, Item 1A.

In [ ]:
from filing_copilot.edgar import EdgarClient
from filing_copilot.filings import build_filing_text, download_company
from filing_copilot.retrieval.labeling import find_passages, gold_entry

def load_filing(ticker: str, fiscal_year: int):
    company = corpus.by_ticker(ticker)
    with EdgarClient(settings) as client:
        documents = download_company(client, company, years=3).documents
    document = next(d for d in documents if d.ref.fiscal_year == fiscal_year)
    return build_filing_text(document, company.name)

syf_2024 = load_filing("SYF", 2024)
passages = find_passages(syf_2024, "largest partners", item="1A", max_results=3)
for p in passages:
    print(f"chars {p.char_start:,}-{p.char_end:,} · {p.document}\n{p.text[:500]}\n")
print(yaml.safe_dump([gold_entry(syf_2024, passages[0])], sort_keys=False, width=1000))

## C.3 · Why labels are spans, not chunk IDs

The two chunkers cut the same filing at different places. A label on a `chunk_id` could
only ever score one of them. Here the passage from C.2 is treated as gold (in memory
only, **nothing is written**), and the same question runs against both indexes. A chunk
earns the span's grade if it covers at least `MIN_OVERLAP_CHARS` of it, or all of it when
the span is shorter.

Different chunk IDs, same passage, both credited. That is what makes the Phase 5 A/B fair.

In [ ]:
from filing_copilot.retrieval.metrics import MIN_OVERLAP_CHARS, GradedSpan, grade_of, overlap

demo_span = GradedSpan(syf_2024.ref.accession, passages[0].char_start, passages[0].char_end, grade=2)
r001 = next(q for q in eval_questions if q.id == "r001")
print(f"gold span {demo_span.char_start:,}-{demo_span.char_end:,} ({demo_span.length} chars); "
      f"MIN_OVERLAP_CHARS = {MIN_OVERLAP_CHARS}\n")
for chunker in CHUNKERS:
    hits = hybrid(r001.question, chunker=chunker, k=10)
    display(pd.DataFrame(
        {"rank": h.rank, "chunk_id": h.chunk_id, "span": f"{h.char_start:,}-{h.char_end:,}",
         "shared chars": overlap(h.char_start, h.char_end, demo_span.char_start, demo_span.char_end)
                         if h.accession == demo_span.accession else 0,
         "grade": grade_of(h, [demo_span])}
        for h in hits
    ).set_index("rank").style.set_caption(chunker))

## C.4 · Run the grid

This is exactly `fc eval-retrieval --strategy both`: 3 retrievers × 2 chunkers × 2
strategies. `global` is the question text alone, which is what the raw retriever does.
`scoped` applies each question's recorded scope as filters, and uses fan-out for peer
questions (Checkpoint E).

Metrics are means over **labelled** questions. `unjudged@10` is the mean number of top-10
hits covering no judged span. While it is high, the other numbers undercount, because an
unlabelled relevant passage scores as a miss. That is what adjudication (C.5) fixes.

In [ ]:
from filing_copilot.retrieval.evaluate import grid, live_search_fn, run_eval, summarize, win_loss, Config

dev = [q for q in eval_questions if q.split == "dev"]
configs = grid(list(CHUNKERS)) + grid(list(CHUNKERS), strategy="scoped")
search_fn = live_search_fn(search_client, encoder, INDEX)
results = run_eval(dev, configs, search_fn, judgments)

summary = pd.DataFrame(
    {"config": s.config.label, "labelled": f"{s.labelled}/{s.questions}", "R@5": s.recall_5,
     "R@10": s.recall_10, "MRR": s.mrr, "nDCG@10": s.ndcg_10, "unjudged@10": s.unjudged_10,
     "peer coverage@10": s.peer_coverage_10}
    for s in summarize(results)
).set_index("config")
display(summary.round(3))

## C.5 · Adjudication: blind, pooled, shuffled

Gold comes from reading one place in a filing. The same material-weakness language also
appears in Item 9A, a risk factor and the auditor's report. A retriever that finds the
unlabelled copy is scored as wrong. The fix, TREC's *pooling*, is `fc adjudicate`:

1. every configuration's top 10 for a labelled question, pooled;
2. anything already judged removed; the same passage from different chunkers merged;
3. **shuffled**, and shown **without** the system that found it, because knowing it
   biases the judgment toward the system you expect to win;
4. graded 0/1/2 per the rubric. Positives can be narrowed to the answering paragraph.

`PoolItem` has no field for the configuration, so the blindness is structural rather than
a matter of discipline. Below, r001's pool uses the in-memory demo span as its only gold.

In [ ]:
from dataclasses import replace as dc_replace
from filing_copilot.retrieval.adjudicate import build_pool, render
from filing_copilot.retrieval.evalset import GoldSpan

demo_r001 = dc_replace(r001, gold=(GoldSpan(demo_span.accession, demo_span.char_start,
                                            demo_span.char_end, 2, syf_2024.text_sha256),))
pool_results = {c: search_fn(demo_r001, c, 10) for c in grid(list(CHUNKERS))}
pool = build_pool(demo_r001, pool_results, judgments)
print(f"{sum(len(h) for h in pool_results.values())} hits across 6 configurations -> "
      f"{len(pool)} unjudged, deduplicated passages to judge\n")
print(render(pool[0], demo_r001)[:900])

## C.6 · Error analysis

For each question a hybrid configuration missed (recall@10 < 1 or MRR < 0.5), record *why*
in `failures.yaml`, using the fixed taxonomy in that file's header. The cause ×
configuration table, not the average, decides what to change next. Then: **change one
thing, rerun dev, record the delta.**

In [ ]:
from filing_copilot.retrieval.evaluate import failure_table, untagged_failures

tags = load_failures(EVALS / "failures.yaml")
table = failure_table(tags)
display(pd.DataFrame(table).T.fillna(0).astype(int) if table else "no failure tags yet")
for config in (Config("hybrid", "item_aware"), Config("hybrid", "fixed_window")):
    todo = untagged_failures(results, tags, config)
    print(f"{config.label}: {len(todo)} untagged miss(es) {todo}")

---
# Checkpoint D · Label check, then the chunker A/B

Two questions: how much can the labels be trusted, and which chunker wins?

## D.1 · Label agreement: the noise floor

Several days after labelling, `fc adjudicate --recheck` re-grades a random 15% of judged
spans **without showing the original grade**. Agreement between the two passes bounds how
precisely any metric can be read. A difference between configurations smaller than what
label noise could produce is "no measurable difference", not a win.

**Cohen's κ, not just percent agreement.** κ corrects for the agreement chance alone would
produce. A labeller who grades almost everything 2 agrees with themselves most of the time
by luck:

In [ ]:
from filing_copilot.retrieval.adjudicate import agreement
from filing_copilot.retrieval.metrics import cohens_kappa, percent_agreement

lazy_first, lazy_second = [2] * 9 + [1], [2] * 9 + [2]
careful_first, careful_second = [2, 2, 1, 0, 1, 2, 0, 1, 2, 0], [2, 2, 1, 0, 1, 2, 0, 2, 2, 0]
display(pd.DataFrame({
    "percent agreement": [percent_agreement(lazy_first, lazy_second), percent_agreement(careful_first, careful_second)],
    "Cohen's kappa": [cohens_kappa(lazy_first, lazy_second), cohens_kappa(careful_first, careful_second)],
}, index=["almost always '2'", "uses all three grades"]).round(2))

measured = agreement(eval_questions, judgments, load_rechecks(EVALS / "rechecks.yaml"))
print("your recheck:", "none yet -- run `fc adjudicate --recheck` a few days after labelling"
      if measured is None else f"{measured.pairs} pairs, {measured.percent:.0%}, kappa {measured.kappa:.2f}")

## D.2 · The A/B: retriever × chunker

The "done when" for Stage 4: what hybrid gains over vector-only, and which chunker wins, in
numbers. Each metric is pivoted into a table of retriever × chunker, followed by
per-question win/loss counts, because with ~30 questions one question can move a mean.

**Caveat before reading it.** The chunkers differ in chunk count and size (below).
Chunk-level recall@k can favor whichever puts more gold text into fewer chunks. The span
rule scores both chunkers against the same passages, and k is fixed, but the difference in
size is still worth stating next to any verdict.

In [ ]:
sizes = pd.DataFrame(
    {"chunks": len(rows_), "median chars": int(pd.Series([r.char_end - r.char_start for r in rows_]).median()),
     "chunks with no item": sum(not r.items for r in rows_)}
    for rows_ in manifests.values()
).set_axis(list(manifests))
display(sizes)

global_results = [r for r in results if r.config.strategy == "global"]
long = pd.DataFrame(
    {"mode": s.config.mode, "chunker": s.config.chunker, "nDCG@10": s.ndcg_10, "R@10": s.recall_10, "MRR": s.mrr}
    for s in summarize(global_results)
)
if long["nDCG@10"].notna().any():
    for metric in ("nDCG@10", "R@10", "MRR"):
        display(long.pivot(index="mode", columns="chunker", values=metric).round(3).style.set_caption(metric))
    for chunker in CHUNKERS:
        for other in ("vector", "bm25"):
            wl = win_loss(global_results, Config("hybrid", chunker), Config(other, chunker))
            print(f"hybrid vs {other:<6} on {chunker:<12} wins {wl.wins}  losses {wl.losses}  ties {wl.ties}")
    for mode in ("vector", "bm25", "hybrid"):
        wl = win_loss(global_results, Config(mode, "item_aware"), Config(mode, "fixed_window"))
        print(f"{mode:<6} item_aware vs fixed_window  wins {wl.wins}  losses {wl.losses}  ties {wl.ties}")
else:
    print("No labelled questions yet -- the A/B needs gold spans (Checkpoint C).")

## D.3 · The blind split, run once

Six questions (r006, r009, r014, r015, r017, r025) have been held out since before any
search code existed. Run them **once**, with the configuration the dev split chose, as a
check on the result, not to make a decision:

```
fc eval-retrieval --split blind
```

Each blind run is logged to `data/processed/eval/blind_runs.log`, and the command warns if
the split has been run before. It is not run from this notebook, so re-running the notebook
cannot spend it.

## D.4 · Recording the verdict

When the grid has labelled numbers and the recheck has a κ:
- **ADR-0002** (chunking): which chunker won, by how much, the win/loss counts, and the
  noise floor. Also the structural result from Checkpoint E, which holds regardless: only
  `item_aware` can be filtered by item.
- **ADR-0010** (RRF): what hybrid gained over vector-only, post-adjudication.

---
# Checkpoint E · Filters and strategies

Two questions no global top-k can answer well, however good the ranking:
- **Peer questions** get *concentration*: the wordiest filer takes most of the slots.
- **Screening questions** ("any material weakness?") get *boilerplate*: the phrase in
  every risk-factor section outranks the one Item 9A that matters.

The fixes are metadata filters, `fan_out_peers` and `section_targeted`, in
`retrieval/strategies.py`. First, one mechanism to get right.

## E.1 · Where a kNN filter goes, and why it matters

A filter **inside** the `knn` clause makes OpenSearch search only the matching vectors,
and still return `k`. A `post_filter` trims an *unfiltered* top-k afterwards, and returns
whatever is left, possibly nothing. Neither raises an error.

In [ ]:
from filing_copilot.retrieval import SearchFilters, vector_query

apple_1a = SearchFilters.of(tickers=["AAPL"], items=["1A"])
v = encode_query(encoder, "customer concentration risk")
inside = search_client.search(INDEX["item_aware"], vector_query(v, k=10, filters=apple_1a))
post = search_client.search(INDEX["item_aware"], {**vector_query(v, k=10), "post_filter": apple_1a.to_query()})
print(f"filter inside knn: {len(inside['hits']['hits'])} hits")
print(f"post_filter:       {len(post['hits']['hits'])} hits   <- the global top 10 held no Apple 1A chunks")

## E.2 · Only one chunker can be targeted by section

`fixed_window` is structure-blind: its chunks carry no item labels, so an item filter
matches nothing (see "chunks with no item" in D.2). Section-targeted retrieval, and
Stage 6's section-aligned change detection, are possible **only** with `item_aware`. That
is a structural argument for ADR-0002 that holds whatever the A/B metrics say.

In [ ]:
from filing_copilot.retrieval import section_targeted

for chunker in CHUNKERS:
    hits = section_targeted(search_client, encoder, INDEX[chunker], "material weakness", ["9A"], k=10)
    print(f"{chunker:>12}: {len(hits)} hits in Item 9A")

## E.3 · Section targeting: 'material weakness' across the portfolio

Demo query 7: "Any material weakness or going-concern language across my portfolio?"
Globally, the top 10 is risk-factor boilerplate from a few filers. Scoped to Item 9A, it is
each filer's controls conclusion. That includes the *credible negatives* ("controls were
effective"), which rubric case 6 grades as answers.

In [ ]:
global_mw = hybrid("material weakness", k=10)
targeted_mw = section_targeted(search_client, encoder, INDEX["item_aware"], "material weakness", ["9A"], k=10)
display(pd.DataFrame({
    "global: ticker/items": [f"{h.ticker} {','.join(h.items)}" for h in global_mw],
    "Item 9A: ticker/FY": [f"{h.ticker} FY{h.fiscal_year}" for h in targeted_mw],
    "Item 9A: text": [" ".join(h.text.split())[:90] for h in targeted_mw],
}, index=range(1, 11)))

## E.4 · Peer questions: concentration vs coverage

The "fail first" step: a peer question with a global top 10, then with `fan_out_peers`,
which runs one filtered search per company and interleaves the results by rank. A company
with nothing relevant is reported as **empty**, which is a finding rather than a gap.

In [ ]:
from filing_copilot.retrieval import distinct_issuers, fan_out_peers, issuer_coverage

peer_q = "How do card issuers describe their CECL allowance methodology?"
issuers = ["COF", "SYF", "AXP", "BFH"]
global_top = hybrid(peer_q, k=10)
fanned = fan_out_peers(search_client, encoder, INDEX["item_aware"], peer_q, issuers, n_per_company=3)
print(f"global top 10:  {distinct_issuers(global_top)} issuers, coverage {issuer_coverage(global_top, issuers):.0%}  "
      f"{[h.ticker for h in global_top]}")
top10 = fanned.hits[:10]
print(f"fan-out top 10: {distinct_issuers(top10)} issuers, coverage {issuer_coverage(top10, issuers):.0%}  "
      f"{[h.ticker for h in top10]}   empty: {fanned.empty or 'none'}")

## E.5 · Coverage across the eval's peer questions

The same comparison over every dev peer question, from C.4's run. Peer coverage needs no
relevance labels. It only asks whether each named company appears, so these numbers are
already final.

In [ ]:
peer_rows = [r for r in results if r.coverage_10 is not None]
coverage = pd.DataFrame(
    {"question": r.question, "config": r.config.label, "coverage@10": r.coverage_10} for r in peer_rows
).pivot(index="config", columns="question", values="coverage@10")
coverage["mean"] = coverage.mean(axis=1)
display(coverage.round(2))

---
# Where Stage 4 stands

**Measured, and final without labels:**
- hybrid vs vector vs BM25 behavior on the probes (A, B), with fused provenance;
- peer coverage: global top-k against fan-out (E.5);
- the kNN filter-placement failure (E.1), and the structural chunker result (E.2).

**Waiting on labels** (the human steps in C):
- recall@k, MRR and nDCG@10 per retriever × chunker, win/loss, and unjudged@10;
- the recheck κ, which is the noise floor;
- the ADR-0002 and ADR-0010 verdicts, and the single blind run.

Reference: `docs/stage4_retrieval.md`.